# Identify material changes

Use matched valuation dates and versioned evidence keys. Signals are triage flags, not binding recommendations.


In [ ]:
import sys
from pathlib import Path
BASE = Path('/lakehouse/default/Files/underwriting_demo')
sys.path.insert(0, str(BASE / 'src'))
from insurance_demo import build_data, validate, column_type, AS_OF, ROOT
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType
TABLE_PREFIX = 'demo_'

def save_rows(name, rows):
    types = {'bit': BooleanType(), 'bigint': LongType(), 'float': DoubleType()}
    schema = StructType([StructField(field, types.get(column_type(field, rows), StringType()), True) for field in rows[0]])
    frame = spark.createDataFrame(rows, schema)
    for field in rows[0]:
        datatype = column_type(field, rows)
        if datatype == 'date':
            frame = frame.withColumn(field, F.to_date(field))
        elif datatype == 'datetime2(6)':
            frame = frame.withColumn(field, F.to_timestamp(field))
    destination = TABLE_PREFIX + name
    if spark.catalog.tableExists(destination):
        properties = {row.key: row.value for row in spark.sql(f'SHOW TBLPROPERTIES {destination}').collect()}
        assert properties.get('delta.columnMapping.mode', 'none') == 'none', 'Use a new managed demo table without column mapping'
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(destination)
    assert spark.table(destination).count() == len(rows)
    return frame

from functools import reduce
features = spark.table(TABLE_PREFIX + 'UnderwritingFeatures')
rules = [('CAT deterioration', 'CATGrowth', 0.25, 'CATRisk'), ('Claims development', 'ClaimsGrowth', 0.25, 'ClaimValuations'), ('Exposure growth', 'TIVGrowth', 0.20, 'Exposures'), ('Engineering deficiencies', 'Engineering', 0.50, 'EngineeringRecommendations'), ('Missing information', 'Missing', 0.50, 'Renewals')]
frames = []
for index, (name, field, threshold, source) in enumerate(rules):
    frames.append(features.filter(F.col(field) >= threshold).select('AccountId', F.concat(F.lit('SIG'), F.col('AccountId'), F.lit(str(index))).alias('SignalId'), F.concat(F.lit('R'), F.substring('AccountId', 2, 3)).alias('RenewalId'), F.lit(name).alias('SignalType'), F.col(field).alias('Value'), F.lit(threshold).alias('Threshold'), F.lit('High').alias('Materiality'), F.lit(source).alias('SourceTable'), F.col('AccountId').alias('EvidenceKey'), F.col('SnapshotDate').alias('DetectedDate')))
signals = reduce(lambda left, right: left.unionByName(right), frames)
signals.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TABLE_PREFIX + 'RenewalSignals')
display(signals)
